# 🧬 Pipeline RNA-seq
## HISAT2 → StringTie → DESeq2

**Auteur :** DrArvJ.K.D  
**Date :** 2025-03-21  
**Référence :** Griffith Lab, CSHL 2025

---

### Étapes
1. Contrôle qualité (FastQC + MultiQC)
2. Indexation du génome (HISAT2-build)
3. Alignement (HISAT2)
4. QC post-alignement
5. Quantification (StringTie + featureCounts)
6. Expression différentielle (DESeq2)

# 🟢 Cellule 2 — Configuration (cellule Code)

In [48]:
import os
from pathlib import Path
os.environ["PATH"] = "/opt/conda/envs/rnaseq_env/bin:" + os.environ["PATH"]

# ─── Répertoires ──────────────────────────────────────────────
BASE_DIR  = Path("..")
DATA_RAW  = BASE_DIR / "data" / "raw"
DATA_REF  = BASE_DIR / "data" / "reference"
DATA_BAM  = BASE_DIR / "data" / "processed" / "bam"
RES_QC    = BASE_DIR / "results" / "01_fastqc"
RES_ALIGN = BASE_DIR / "results" / "02_alignment"
RES_ABUND = BASE_DIR / "results" / "03_abundance"
RES_DE    = BASE_DIR / "results" / "04_differential_expression"
RES_FIG   = BASE_DIR / "results" / "05_figures"
LOGS      = BASE_DIR / "logs"

# ─── Paramètres ───────────────────────────────────────────────
N_THREADS = 8
GTF       = DATA_REF / "chr22_with_ERCC92.gtf"
GENOME    = DATA_REF / "chr22_with_ERCC92.fa"
INDEX     = DATA_REF / "hisat2_index" / "chr22"

# ─── Créer les dossiers manquants ─────────────────────────────
for d in [DATA_RAW, DATA_REF, DATA_BAM, RES_QC, RES_ALIGN,
          RES_ABUND, RES_DE, RES_FIG, LOGS, INDEX.parent]:
    d.mkdir(parents=True, exist_ok=True)

print("✅ Répertoires prêts")
print(f"   Base    : {BASE_DIR.resolve()}")
print(f"   Threads : {N_THREADS}")

✅ Répertoires prêts
   Base    : /data/jwd07/pulsar_staging/98584065/working/jupyter/galaxy_inputs/rnaseq_analysis
   Threads : 8


# 🟢 Cellule 3 — Téléchargement des données de test (cellule Code)

In [18]:
# fichier fastq
!wget -q -P{DATA_RAW} http://genomedata.org/rnaseq-tutorial/HBR_UHR_ERCC_ds_5pc.tar

In [21]:
#decompresse les fichiers fastqc
!tar xf HBR_UHR_ERCC_ds_5pc.tar -C {DATA_RAW}

In [14]:
# TELECHARGER chr22_with_ERCC92.fa
!wget -q -P{DATA_REF} http://genomedata.org/rnaseq-tutorial/fasta/GRCh38/chr22_with_ERCC92.fa

## TELECHARGER Homo_sapiens.GRCh378_chr22_ERCC.gtf
!wget -q -P{DATA_REF} http://genomedata.org/rnaseq-tutorial/annotations/GRCh38/chr22_with_ERCC92.gtf


In [3]:
#regarde si le telechargement a ete bien fait
!ls -lh {DATA_RAW}
!ls -lh {DATA_REF}

total 223M
-rw-r--r--. 1 jovyan users 6.4M Nov  6  2014 HBR_Rep1_ERCC-Mix2_Build37-ErccTranscripts-chr22.read1.fastq.gz
-rw-r--r--. 1 jovyan users 6.7M Nov  6  2014 HBR_Rep1_ERCC-Mix2_Build37-ErccTranscripts-chr22.read2.fastq.gz
-rw-r--r--. 1 jovyan users 7.7M Nov  6  2014 HBR_Rep2_ERCC-Mix2_Build37-ErccTranscripts-chr22.read1.fastq.gz
-rw-r--r--. 1 jovyan users 8.1M Nov  6  2014 HBR_Rep2_ERCC-Mix2_Build37-ErccTranscripts-chr22.read2.fastq.gz
-rw-r--r--. 1 jovyan users 6.9M Nov  6  2014 HBR_Rep3_ERCC-Mix2_Build37-ErccTranscripts-chr22.read1.fastq.gz
-rw-r--r--. 1 jovyan users 7.3M Nov  6  2014 HBR_Rep3_ERCC-Mix2_Build37-ErccTranscripts-chr22.read2.fastq.gz
-rw-r--r--. 1 jovyan users 112M Oct 23  2018 HBR_UHR_ERCC_ds_5pc.tar
-rw-r--r--. 1 jovyan users  13M Nov  6  2014 UHR_Rep1_ERCC-Mix1_Build37-ErccTranscripts-chr22.read1.fastq.gz
-rw-r--r--. 1 jovyan users  14M Nov  6  2014 UHR_Rep1_ERCC-Mix1_Build37-ErccTranscripts-chr22.read2.fastq.gz
-rw-r--r--. 1 jovyan users 9.7M Nov  6  2014 UHR

# 🟢 Cellule 4 — FastQC (contrôle qualité des reads)

In [8]:
!fastqc {DATA_RAW}/*.fastq.gz \
        --outdir {RES_QC} \
        --threads {N_THREADS} \
        2>&1 | tail -5

Approx 80% complete for UHR_Rep3_ERCC-Mix1_Build37-ErccTranscripts-chr22.read2.fastq.gz
Approx 85% complete for UHR_Rep3_ERCC-Mix1_Build37-ErccTranscripts-chr22.read2.fastq.gz
Approx 90% complete for UHR_Rep3_ERCC-Mix1_Build37-ErccTranscripts-chr22.read2.fastq.gz
Approx 95% complete for UHR_Rep3_ERCC-Mix1_Build37-ErccTranscripts-chr22.read2.fastq.gz
Analysis complete for UHR_Rep3_ERCC-Mix1_Build37-ErccTranscripts-chr22.read2.fastq.gz


# 🟢 Cellule 5 — MultiQC (agrège tous les rapports FastQC)

In [9]:
# MultiQC agrège tous les rapports FastQC en un seul rapport HTML
# - cherche dans RES_QC tous les fichiers FastQC reconnus
# - fusionne en un seul rapport HTML
# - --force : écrase si le rapport existe déjà
!multiqc {RES_QC} \
         --outdir {RES_QC} \
         --filename multiqc_report.html \
         --force \
         2>&1 | tail -5

|         searching | ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 24/24  0m  
|            fastqc | Found 12 reports
|           multiqc | Report      : ../results/01_fastqc/multiqc_report.html
|           multiqc | Data        : ../results/01_fastqc/multiqc_report_data
|           multiqc | MultiQC complete


# 🟢 Cellule 6 — Indexation du génome (HISAT2-build)

In [10]:
# Indexation du génome de référence avec HISAT2
# - opération à faire une seule fois par génome
# - crée ~8 fichiers .ht2 dans le dossier hisat2_index/
!hisat2-build \
    {GENOME} \
    {INDEX} \
    -p {N_THREADS} \
    2>&1 | tail -5

    gbwtTotLen: 13080896
    gbwtTotSz: 13080896
    reverse: 0
    linearFM: Yes
Total time for call to driver() for forward index: 00:01:02


In [11]:
# Vérification des fichiers d'index créés par HISAT2
!ls -lh {DATA_REF}/hisat2_index/

total 62M
-rw-r--r--. 1 jovyan users  17M Mar 21 19:46 chr22.1.ht2
-rw-r--r--. 1 jovyan users 9.4M Mar 21 19:46 chr22.2.ht2
-rw-r--r--. 1 jovyan users 1.3K Mar 21 19:45 chr22.3.ht2
-rw-r--r--. 1 jovyan users 9.4M Mar 21 19:45 chr22.4.ht2
-rw-r--r--. 1 jovyan users  18M Mar 21 19:46 chr22.5.ht2
-rw-r--r--. 1 jovyan users 9.6M Mar 21 19:46 chr22.6.ht2
-rw-r--r--. 1 jovyan users   12 Mar 21 19:45 chr22.7.ht2
-rw-r--r--. 1 jovyan users    8 Mar 21 19:45 chr22.8.ht2


# 🟢 Cellule 7 — Alignement test UHR_Rep1 sans strandness

In [12]:
# Alignement test sur un seul échantillon SANS --rna-strandness
# But : déterminer le strandness avant d'aligner tous les échantillons
os.system(
    f"hisat2 -p {N_THREADS} --dta "
    f"-x {INDEX} "
    f"-1 {DATA_RAW}/UHR_Rep1_ERCC-Mix1_Build37-ErccTranscripts-chr22.read1.fastq.gz "
    f"-2 {DATA_RAW}/UHR_Rep1_ERCC-Mix1_Build37-ErccTranscripts-chr22.read2.fastq.gz "
    f"2>{LOGS}/UHR_Rep1_test.log | "
    f"samtools sort -@ {N_THREADS} -o {DATA_BAM}/UHR_Rep1_test.sorted.bam"
)
os.system(f"samtools index {DATA_BAM}/UHR_Rep1_test.sorted.bam")
print("✅ BAM test créé")

[bam_sort_core] merging from 0 files and 8 in-memory blocks...


✅ BAM test créé


In [41]:
# On s'assure que l'alignement se fait bien
!samtools flagstat {DATA_BAM}/UHR_Rep1_test.sorted.bam

/bin/bash: line 1: samtools: command not found


In [45]:
!which gtfToGenePred

/opt/conda/envs/rnaseq_env/bin/gtfToGenePred


In [37]:
# Conversion du GTF en BED (format requis par infer_experiment.py)
# gtfToGenePred → genePredToBed : deux conversions successives
!gtfToGenePred {GTF} /tmp/chr22.genePred
!genePredToBed /tmp/chr22.genePred {DATA_REF}/chr22_with_ERCC92.bed

# Vérification du fichier BED créé
!wc -l {DATA_REF}/chr22_with_ERCC92.bed

4564 ../data/reference/chr22_with_ERCC92.bed


In [39]:
# Vérifier le format des chromosomes dans le BAM
!samtools view {DATA_BAM}/UHR_Rep1_test.sorted.bam | head -1 | cut -f3

/bin/bash: line 1: samtools: command not found


In [50]:
!ls /opt/conda/envs/rnaseq_env/bin/samtools

ls: cannot access '/opt/conda/envs/rnaseq_env/bin/samtools': No such file or directory


# 🟢 Vérifie le strandness avec infer_experiment.py
est un outil du package RSeQC. Il répond à la question : "Dans quel sens mes reads ont-ils été séquencés ?"

In [38]:
# infer_experiment.py analyse 200 000 reads du BAM
# et compare leur orientation avec le GTF
# pour deviner si la librairie est stranded ou non
!infer_experiment.py \
    -r {DATA_REF}/chr22_with_ERCC92.bed \
    -i {DATA_BAM}/UHR_Rep1_test.sorted.bam \
    2>&1 | tail -20

/bin/bash: line 1: infer_experiment.py: command not found


462957 + 0 in total (QC-passed reads + QC-failed reads)
454784 + 0 primary
8173 + 0 secondary
0 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
461882 + 0 mapped (99.77% : N/A)
453709 + 0 primary mapped (99.76% : N/A)
454784 + 0 paired in sequencing
227392 + 0 read1
227392 + 0 read2
451596 + 0 properly paired (99.30% : N/A)
452916 + 0 with itself and mate mapped
793 + 0 singletons (0.17% : N/A)
4 + 0 with mate mapped to a different chr
4 + 0 with mate mapped to a different chr (mapQ>=5)


# 🟢 Cellule 7 — Alignement avec HISAT2

In [ ]:
# Échantillons à traiter
samples = [
    "UHR_Rep1_ERCC-Mix1_Build37-ErccTranscripts-chr22",
    "UHR_Rep2_ERCC-Mix1_Build37-ErccTranscripts-chr22",
    "UHR_Rep3_ERCC-Mix1_Build37-ErccTranscripts-chr22",
    "HBR_Rep1_ERCC-Mix2_Build37-ErccTranscripts-chr22",
    "HBR_Rep2_ERCC-Mix2_Build37-ErccTranscripts-chr22",
    "HBR_Rep3_ERCC-Mix2_Build37-ErccTranscripts-chr22",
]

In [ ]:
# Alignement + tri + indexation de tous les échantillons
for s in samples:

    # Chemins des fichiers d'entrée et sortie
    r1         = f"{DATA_RAW}/{s}.read1.fastq.gz"
    r2         = f"{DATA_RAW}/{s}.read2.fastq.gz"
    sorted_bam = f"{DATA_BAM}/{s}.sorted.bam"
    log        = f"{LOGS}/{s}_hisat2.log"

    print(f"\n=== Traitement de {s} ===")

    # HISAT2 → samtools sort en un seul pipe (pas de fichier SAM intermédiaire)
    os.system(
        f"hisat2 -p {N_THREADS} --dta --rna-strandness RF "
        f"-x {INDEX} -1 {r1} -2 {r2} 2>{log} | "
        f"samtools sort -@ {N_THREADS} -o {sorted_bam}"
    )

    # Indexation du BAM trié
    os.system(f"samtools index {sorted_bam}")

    print(f"✅ {s} → {sorted_bam}")

print("\n✅ Tous les échantillons sont alignés !")